# Feature engineering - advanced data preparation pipeline 

## Libraries

In [60]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
# from tabpfn import TabPFNClassifier

## Data

In [61]:
data_path = ''
pd.set_option('display.max_columns', None)

SampleSubmissionStage2 = pd.read_csv(join('../../data', 'SampleSubmissionStage2.csv'))

MenTest = pd.read_csv(join(data_path, "MenTest.csv"), index_col=0)
MenTrain = pd.read_csv(join(data_path, "MenTrain.csv"), index_col=0)

WomenTest = pd.read_csv(join(data_path, "WomenTest.csv"), index_col=0)
WomenTrain = pd.read_csv(join(data_path, 'WomenTrain.csv'), index_col=0)

# MenTest and WomenTest have the same columns and the same
# 'Season', 'T1_TeamID', 'T1_Score', 'T2_TeamID', 'T2_Score', 'location'
# values in the same order, but Men have only daata for men teams and NaNs in other rows.
# The same for WomenTest.
# Test below fills the NaNs with values from the data frame, where values are present.
Test = MenTest.combine_first(WomenTest)

## Data preparation pipeline

In [62]:
def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[6:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

## Get data to use models on for testing

In [63]:
THE_LAST_YEAR = 2024
# I will be using train and test from train dataset
x_train_men, y_train_men = x_y_from_data_frame(MenTrain[MenTrain['Season']<THE_LAST_YEAR])
x_train_women, y_train_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']<THE_LAST_YEAR])
x_test_men, y_test_men = x_y_from_data_frame(MenTrain[MenTrain['Season']==THE_LAST_YEAR])
x_test_women, y_test_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']==THE_LAST_YEAR])

In [64]:
x_test_women

array([[  20.8       ,   52.4       ,    4.4       , ..., 1424.46861094,
        1612.00052755, 1612.00052755],
       [  22.75      ,   60.75      ,    6.        , ..., 1890.68195589,
        1612.00052755, 1612.00052755],
       [  26.4       ,   62.4       ,    3.6       , ..., 1881.97790007,
        1612.00052755, 1612.00052755],
       ...,
       [  27.83333333,   57.66666667,    7.33333333, ..., 2306.63399602,
        1612.00052755, 1612.00052755],
       [  24.        ,   61.5       ,    3.66666667, ..., 2540.7036061 ,
        1612.00052755, 1612.00052755],
       [  34.66666667,   67.83333333,   14.83333333, ..., 2540.7036061 ,
        1612.00052755, 1612.00052755]])

# Training models

## Testing the model

#### For men:
GradientBoostingClassifier(learning_rate=0.01, max_depth=8,
                           min_samples_leaf=0.045, n_estimators=987,
                           random_state=2137, subsample=0.4)
Brier score: 0.1887166502270038

#### For women:
LogisticRegression(C=0.08545751829602459, class_weight='balanced',
                   max_iter=10000, penalty='l1', random_state=2137,
                   solver='liblinear')
Brier score: 0.12700467150133687

In [65]:
gbc_men = GradientBoostingClassifier(learning_rate=0.01, max_depth=8,
                                     min_samples_leaf=0.045, n_estimators=987,
                                     random_state=2137, subsample=0.4)
lr_women = LogisticRegression(C=0.08545751829602459, class_weight='balanced',
                              max_iter=10000, penalty='l1', random_state=2137,
                              solver='liblinear')


# Create Submission

In [66]:
data_path = ''
pd.set_option('display.max_columns', None)

SampleSubmissionStage2 = pd.read_csv(join('../../data', 'SampleSubmissionStage2.csv'))

MenTest = pd.read_csv(join(data_path, "MenTest.csv"), index_col=0)
MenTrain = pd.read_csv(join(data_path, "MenTrain.csv"), index_col=0)

WomenTest = pd.read_csv(join(data_path, "WomenTest.csv"), index_col=0)
WomenTrain = pd.read_csv(join(data_path, 'WomenTrain.csv'), index_col=0)

# MenTest and WomenTest have the same columns and the same
# 'Season', 'T1_TeamID', 'T1_Score', 'T2_TeamID', 'T2_Score', 'location'
# values in the same order, but Men have only daata for men teams and NaNs in other rows.
# The same for WomenTest.
# Test below fills the NaNs with values from the data frame, where values are present.
#Test = MenTest.combine_first(WomenTest)

In [67]:
x_train_men, y_train_men = x_y_from_data_frame(MenTrain)
x_train_men

array([[  18.75      ,   47.5       ,    4.375     , ..., 1435.94019103,
        1547.44974943, 1577.304377  ],
       [  27.57142857,   55.85714286,    6.57142857, ..., 1607.95301979,
        2013.80134003, 1727.45495265],
       [  23.25      ,   47.5       ,    7.75      , ..., 1814.14842004,
        2086.49466787, 1888.17228591],
       ...,
       [  31.5       ,   67.66666667,    8.33333333, ..., 2221.4655707 ,
        2068.95188099, 2257.47883928],
       [  27.4       ,   58.4       ,    6.7       , ..., 2178.11347661,
        1981.34106167, 2191.54686187],
       [  26.16666667,   55.66666667,    7.5       , ..., 2221.4655707 ,
        2191.54686187, 2257.47883928]])

In [68]:
x_test_men, y_test_men = x_y_from_data_frame(MenTest.dropna(subset=['T1_FGM']))
x_train_women, y_train_women = x_y_from_data_frame(WomenTrain)
x_test_women, y_test_women = x_y_from_data_frame(WomenTest.dropna(subset=['T1_FGM']))
x_test_women

array([[  22.83333333,   53.5       ,    8.        , ..., 1462.51858311,
        1612.00052755, 1612.00052755],
       [  22.83333333,   53.5       ,    8.        , ..., 1311.67283777,
        1612.00052755, 1612.00052755],
       [  22.83333333,   53.5       ,    8.        , ..., 2042.93340808,
        1612.00052755, 1612.00052755],
       ...,
       [  21.16666667,   53.83333333,    6.33333333, ..., 1241.65868077,
        1612.00052755, 1612.00052755],
       [  21.16666667,   53.83333333,    6.33333333, ..., 1361.9865805 ,
        1612.00052755, 1612.00052755],
       [  21.        ,   49.5       ,    5.5       , ..., 1361.9865805 ,
        1612.00052755, 1612.00052755]])

In [69]:
# I will fill the NaNs (seeds) with 0. It doesn't matter, since those teams won't play in the tournament anyway.
x_test_men = np.nan_to_num(x_test_men, nan=0)
x_test_women = np.nan_to_num(x_test_women, nan=0)

In [70]:
gbc_men.fit(x_train_men, y_train_men)
lr_women.fit(x_train_women, y_train_women)
#brier score
print(f"Brier score mens model: {brier_score_loss(y_train_men, gbc_men.predict_proba(x_train_men)[:, 1])}")
print(f"Brier score womens model: {brier_score_loss(y_train_women, lr_women.predict_proba(x_train_women)[:, 1])}")

Brier score mens model: 0.11476739297248263
Brier score womens model: 0.1252253374240375


In [71]:
predictions_men = gbc_men.predict_proba(x_test_men)[:, 1]
predictions_men

array([0.3253074 , 0.35364107, 0.07596745, ..., 0.59855798, 0.70470254,
       0.59882864])

In [72]:
predictions_women = gbc_men.predict_proba(x_test_women)[:, 1]
predictions_women

array([0.53041909, 0.54468353, 0.26610673, ..., 0.52145021, 0.56956971,
       0.55290001])

In [73]:
Submission = SampleSubmissionStage2.copy()
Submission

,ID,Pred
0,2025_1101_1102,0.5
1,2025_1101_1103,0.5
2,2025_1101_1104,0.5
3,2025_1101_1105,0.5
4,2025_1101_1106,0.5
...,...,...
131402,2025_3477_3479,0.5
131403,2025_3477_3480,0.5
131404,2025_3478_3479,0.5
131405,2025_3478_3480,0.5


In [74]:
#create new ndarray with predictions: first prediction_men, then prediction_women
predictions = np.concatenate((predictions_men, predictions_women))
predictions

array([0.3253074 , 0.35364107, 0.07596745, ..., 0.52145021, 0.56956971,
       0.55290001])

In [75]:
Submission

,ID,Pred
0,2025_1101_1102,0.5
1,2025_1101_1103,0.5
2,2025_1101_1104,0.5
3,2025_1101_1105,0.5
4,2025_1101_1106,0.5
...,...,...
131402,2025_3477_3479,0.5
131403,2025_3477_3480,0.5
131404,2025_3478_3479,0.5
131405,2025_3478_3480,0.5


In [76]:
Submission['Pred'] = predictions

In [77]:
Submission

,ID,Pred
0,2025_1101_1102,0.325307
1,2025_1101_1103,0.353641
2,2025_1101_1104,0.075967
3,2025_1101_1105,0.328793
4,2025_1101_1106,0.303848
...,...,...
131402,2025_3477_3479,0.390948
131403,2025_3477_3480,0.431432
131404,2025_3478_3479,0.521450
131405,2025_3478_3480,0.569570


In [78]:
Submission.to_csv("gbcmen_lrwomen.csv", index=False)

In [79]:
pd.read_csv(join(data_path, "gbcmen_lrwomen.csv"))

,ID,Pred
0,2025_1101_1102,0.325307
1,2025_1101_1103,0.353641
2,2025_1101_1104,0.075967
3,2025_1101_1105,0.328793
4,2025_1101_1106,0.303848
...,...,...
131402,2025_3477_3479,0.390948
131403,2025_3477_3480,0.431432
131404,2025_3478_3479,0.521450
131405,2025_3478_3480,0.569570
